# Clustering with K-means and DBSCAN
In this demonstration, we will test the k-means and DBSCAN algorithms with network traffic. Specifically, we will use such clustering techniques to group network flows into clusters in an unsupervised fashion.

We will use a dataset of benign and various DDoS attacks from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 1000 packets:

| Feature nr.         | Feature Name |
|---------------------|---------------------|
| 00 | timestamp (mean IAT) | 
| 01 | packet_length (mean)| 
| 02 | IP_ttl (mean) |
| 03 | IP_flags_df (sum) |
| 04 | IP_flags_mf (sum) |
| 05 | IP_flags_rb (sum) | 
| 06 | IP_frag_off (sum) |
| 07 | TCP_length (mean) |
| 08 | TCP_flags_ack (sum) |
| 09 | TCP_flags_cwr (sum) |
| 10 | TCP_flags_ece (sum) |
| 11 | TCP_flags_fin (sum) |
| 12 | TCP_flags_push (sum) |
| 13 | TCP_flags_reset (sum) |
| 14 | TCP_flags_syn (sum) |
| 15 | TCP_flags_urg (sum) |
| 16 | TCP_window_size (mean) |
| 17 | UDP_length (mean) |
| 18 | ICMP_type (mean) |
| 19 | ICMP_code (mean) |
| 20 | Packets (counter)|

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import numpy as np
import matplotlib.pyplot as plt
from util_functions import *
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

DATASET_FOLDER = './DOS2019'

X_train, y_train =  load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
# Classes available in the dataset
traffic_classes = {'benign': 0, 'dns': 1, 'syn': 2, 'udplag': 3, 'webddos': 4}

# Classes used in the demonstration
target_classes = ['dns','syn'] 

traffic_class_0 = traffic_classes[target_classes[0]]
traffic_class_1 = traffic_classes[target_classes[1]]

# Extract only the target classes from the dataset from the dataset
X_class = {}
y_class = {}

for cls in range(5):
    mask = (y_train == cls)
    X_class[cls] = X_train[mask]
    y_class[cls] = y_train[mask]

# Combine samples from only two classes
X_train_2_classes = np.concatenate((X_class[traffic_class_0], X_class[traffic_class_1]), axis=0)
# Combine the corresponding labels
y_train_2_classes = np.concatenate((y_class[traffic_class_0], y_class[traffic_class_1]), axis=0)

SEED=1
feature_names = get_feature_names(flatten=True)




# Dimensionality reduction with PCA
PCA is a linear dimensionality reduction technique that transforms the data into a new coordinate system such that the greatest variance by any projection of the data comes to lie on the first coordinate (called the first principal component), the second greatest variance on the second coordinate, and so on. PCA is often used to reduce the dimensionality of large datasets, increasing interpretability while minimizing information loss.

In this demo we will use PCA to reduce the dimensionality of the dataset from 21 features to 2 features, and then we will apply k-means clustering on the reduced dataset.

In [ ]:
pca_2d = PCA(n_components=2)
X_train_2D = pca_2d.fit_transform(X_train_2_classes)


# Dataset analysis
We first plot the distribution of the features to understand why the effectiveness of clustering strongly depends on the choice of the input features. If the selected features do not capture the underlying structure of the data, samples belonging to different classes may overlap, making it difficult or even impossible for the clustering algorithm to identify meaningful groups. Conversely, choosing informative and discriminative features results in well-separated clusters, enabling the algorithm to distinguish between different classes more effectively.

In this demo, we use the features obtained with dimensionality reduction with PCA, which are linear combinations of the original features. The PCA transformation is designed to maximize the variance in the data, which can help to separate different classes in the reduced feature space.

In [ ]:

for i in range(X_train_2D.shape[1]):
    class0 = X_train_2D[y_train_2_classes == traffic_class_0, i]
    class1 = X_train_2D[y_train_2_classes == traffic_class_1, i]
    label=traffic_classes[target_classes[0]]

    plt.figure(figsize=(10, 5))
    plt.hist(class0, bins=50, alpha=0.5, label=traffic_classes[target_classes[0]], color='blue', density=True)
    plt.hist(class1, bins=50, alpha=0.5, label=traffic_classes[target_classes[1]], color='orange', density=True)
    plt.title(f'PCA Feature: {i}')
    plt.xlabel('Feature Value')
    plt.ylabel('Density')
    plt.legend()
    plt.grid()
    plt.show()

## Plot the points in a 2D feature space
For visualizazion purposes, we select only two features to plot the distribution of benign and ddos traffic in a 2D space.

In [ ]:
# Select only the sencond and the third features for visualization purposes
feature_1 = 'PCA Feature 0'
feature_2 = 'PCA Feature 1'

# Visualize the three classes in a 2D scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(X_train_2D[y_train_2_classes == traffic_class_0, 0], X_train_2D[y_train_2_classes == traffic_class_0, 1], label=target_classes[0], marker='o', color='blue', edgecolor='k', s=100)
plt.scatter(X_train_2D[y_train_2_classes == traffic_class_1, 0], X_train_2D[y_train_2_classes == traffic_class_1, 1], label=target_classes[1], marker='s', color='yellow', edgecolor='k', s=100)
plt.xlabel(feature_1)
plt.ylabel(feature_2)
plt.legend()        

## Clustering with k-Means

In [ ]:
k = 2 #len(target_names)
kmeans = KMeans(n_clusters=k, random_state=SEED) # The SEED is used for reproducibility, ensuring that the same initial centroids are chosen each time the algorithm is run.
y_pred_kmeans = kmeans.fit_predict(X_train_2D)

In [ ]:
# Visualize the clusters in a 2D scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(X_train_2D[:, 0], X_train_2D[:, 1], c=y_pred_kmeans, cmap='viridis', marker='o', edgecolor='k', s=100)
plt.title('KMeans Clustering')
plt.xlabel(feature_1)
plt.ylabel(feature_2)       
plt.colorbar(label='Cluster Label')
plt.show()

## K-means accuracy
k-means assign a cluster label to each data point, but these labels do not necessarily correspond to the true class labels. To evaluate the performance of K-Means, we can use metrics such as Adjusted Rand Index (ARI), which compare the predicted cluster labels with the true class labels without requiring a one-to-one mapping between them.

In [ ]:
from sklearn.metrics import adjusted_rand_score 

ari = adjusted_rand_score(y_train_2_classes, y_pred_kmeans)

print(f"Adjusted Rand Index (ARI): {ari}")

## Evaluation metrics: WCSS
Now, we evaluate the quality of the clusters with the Within-Cluster Sum of Squares (WCSS) metric. WCSS is a measure of how tightly the clusters are packed. It is calculated as the sum of squared distances between each point and the centroid of its assigned cluster. A lower WCSS indicates that the clusters are more compact and well-defined.

In [ ]:
wcss = kmeans.inertia_
print(f"Within-Cluster Sum of Squares (WCSS): {wcss}")

## Optimal number of clusters with the "elbow" method

In [ ]:
# Optimal number of clusters using the Elbow Method
wcss_values = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=SEED,n_init=5)
    kmeans.fit(X_train_2D)
    wcss_values.append(kmeans.inertia_) 

# Plot the WCSS values to visualize the Elbow Method
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), wcss_values, marker='o')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(range(1, 11))
plt.grid()
plt.show()


## Clustering with DBSCAN
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is a density-based clustering algorithm that groups together points that are closely packed together, while marking points that lie alone in low-density regions as outliers. Unlike k-means, DBSCAN does not require the number of clusters to be specified in advance and can find arbitrarily shaped clusters.

However, DBSCAN has two hyperparameters: `eps`, which defines the maximum distance between two samples for them to be considered as in the same neighborhood, and `min_samples`, which is the number of samples in a neighborhood for a point to be considered as a core point. The choice of these parameters can significantly affect the clustering results.

In [ ]:
dbscan = DBSCAN(eps=0.6, min_samples=10)
y_pred_dbscan = dbscan.fit_predict(X_train_2D)

print (f"DBSCAN Cluster Labels: {np.unique(y_pred_dbscan)}")

# Visualize the clusters in a 2D scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(X_train_2D[:, 0], X_train_2D[:, 1], c=y_pred_dbscan, cmap='viridis', marker='o', edgecolor='k', s=100)
plt.title('DBSCAN Clustering')
plt.xlabel(feature_1)
plt.ylabel(feature_2)       
plt.colorbar(label='Cluster Label')
plt.show()

## DBSCAN accuracy

In [ ]:
ari = adjusted_rand_score(y_train_2_classes, y_pred_dbscan)

print(f"Adjusted Rand Index (ARI): {ari}")